# CHECKING THE DATA FOR LOSS

In [167]:
### LOADS
import pandas as pd

### FUNCTIONS
def load_parquet_to_df(parquet_path, na=False, verbose=True):
    try:
        df = pd.read_parquet(parquet_path)
        if verbose:
            print(f"Successfully loaded Parquet from {parquet_path}")
            print(f"DataFrame shape: {df.shape}")
            print(f"DataFrame N/A counts:\n{df.isna().sum()}\n")
            print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading Parquet from {parquet_path}: {e}")
        return None
    
folder = "../Data/crawl2_files/"

### Pipeline 2: `LOCAL_txt_metrics_extractor.py`
file(s):
* `extracted_metrics_unified.parquet`

In [ ]:
file_pipe_2 = "extracted_metrics_unified.parquet"

pipe_2 = load_parquet_to_df(f"{folder}{file_pipe_2}", na=True, verbose=False)
pipe_2_len = len(pipe_2)
print(f"Length of pipe_2: {pipe_2_len}")

track = "2372465945 2"
track_df = pipe_2[pipe_2["filename"].str.contains(track)]
print(f"Records in pipe_2 containing '{track}': {len(track_df)}")
# renaming "2372465945 2" to "2372465945" in the filename column of pipe_2
pipe_2["filename"] = pipe_2["filename"].str.replace("2372465945 2", "2372465945", regex=False)
track_df_updated = pipe_2[pipe_2["filename"].str.contains("2372465945")]
print(f"Records in pipe_2 containing '2372465945' after update: {len(track_df_updated)}")


pipe_2[pipe_2["filename"].str.contains("2372465945")].head()

Length of pipe_2: 9889


KeyError: 'ID'

In [169]:
# check for duplicates in pipe_2 based on 'filename' column
duplicates_pipe_2 = pipe_2[pipe_2.duplicated(subset=['filename'], keep=False)]
num_duplicates_pipe_2 = len(duplicates_pipe_2)
print(f"Number of duplicate records in pipe_2 based on 'filename': {num_duplicates_pipe_2}")

Number of duplicate records in pipe_2 based on 'filename': 0


### Pipeline 3: `LOCAL_enrich_w_meta.py`
file(s):
* `thesis_meta_all_metrics_except_grade_and_supervisor_LEFT.parquet` 
* `thesis_meta_all_metrics_except_grade_and_supervisor_INNER.parquet`

In [170]:
file_pipe_3 = "thesis_meta_all_metrics_except_grade_and_supervisor_LEFT.parquet"

pipe_3 = load_parquet_to_df(f"{folder}{file_pipe_3}", na=True, verbose=False)
pipe_3_len = len(pipe_3)
print(f"Length of pipe_3: {pipe_3_len}")
print(f"Loss from pipe_2 to pipe_3: {pipe_2_len - pipe_3_len} records")

Length of pipe_3: 15370
Loss from pipe_2 to pipe_3: -5481 records


In [172]:
# listing the recods in pipe_2["filename"] that are not in pipe_3["filename"]
filenames_pipe_2 = set(pipe_2["filename"])
filenames_pipe_3 = set(pipe_3["filename"])
missing_filenames = filenames_pipe_2 - filenames_pipe_3
print(f"Number of records in pipe_2 that are missing in pipe_3 based on 'filename': {len(missing_filenames)}")
# storing the missing filnames in a df with columns "filename" and "ID" where ID is the part of the filename before ".txt"
missing_df = pd.DataFrame({"filename": list(missing_filenames)})
missing_df["ID"] = missing_df["filename"].apply(lambda x: x.split(".txt")[0])
display(missing_df.head())

Number of records in pipe_2 that are missing in pipe_3 based on 'filename': 316


,filename,ID
0,2372438831.txt,2372438831
1,2727253445.txt,2727253445
2,2712624370.txt,2712624370
3,2712810238.txt,2712810238
4,2733712278.txt,2733712278


In [173]:
old_meta_df = pd.read_csv("../Data/crawl2_files/meta_findit/bulk_meta_findit_all_merged.csv", sep=";")
# check how many of the "ID" in missing_df are in old_meta_df["ID"]
missing_df["ID"] = missing_df["ID"].astype(str)
old_meta_df["ID"] = old_meta_df["ID"].astype(str)
missing_in_old_meta = missing_df[missing_df["ID"].isin(old_meta_df["ID"])]
print(f"Number of records in missing_df that are in old_meta_df based on 'ID': {len(missing_in_old_meta)}")
print(f"Total overlap is {len(missing_in_old_meta)} out of {len(missing_df)} missing records, which is {(len(missing_in_old_meta)/len(missing_df))*100:.2f}%")

Number of records in missing_df that are in old_meta_df based on 'ID': 270
Total overlap is 270 out of 316 missing records, which is 85.44%


/var/folders/kb/z559lc5s7jzbc1c4j024j2w40000gn/T/ipykernel_54059/3747235132.py:1: DtypeWarning: Columns (0: DOI, 1: ISBN) have mixed types. Specify dtype option on import or set low_memory=False.
  old_meta_df = pd.read_csv("../Data/crawl2_files/meta_findit/bulk_meta_findit_all_merged.csv", sep=";")


In [174]:
# see rows with missing "Department_new" in pipe_3
missing_dept = pipe_3[pipe_3["Department_new"].isna()]
print(f"Number of records with missing 'Department_new': {len(missing_dept)}")
print("Sample records with missing 'Department_new':")
print(missing_dept[["filename", "ID", "primary_member_id_s", "Department_new"]].head())
display(missing_dept)

Number of records with missing 'Department_new': 1
Sample records with missing 'Department_new':
     filename          ID       primary_member_id_s Department_new
1144      NaN  2720448208  640bd4d195691a3a3e9a3f49            NaN


,abstract_ts,Author,num_authors,Publication Year,primary_member_id_s,Title,Department_new,ID,filename,num_tot_pages,...,num_tables,num_references,total_sentences,total_words,unique_words,avg_sentence_length,avg_word_length,lexical_diversity,flesch_kincaid_grade,handin_month
1144,The aim of this thesis was to examine whether ...,"Norrbacka, Susanna",1,2022,640bd4d195691a3a3e9a3f49,Developing Automation for Microbial Engineering,NaN,2720448208,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [175]:
df_meta = pd.read_csv("../Data/crawl2_files/meta_findit_all_merged.csv", sep=";", encoding="utf-8", low_memory=False)

target = 2738459452
# searching for target in meta_df["ID"]
target_row = df_meta[df_meta["ID"] == target]
if not target_row.empty:
    print(f"Record with ID {target} found in df_meta:")
    display(target_row)
else:
    print(f"Record with ID {target} not found in df_meta.")

FileNotFoundError: [Errno 2] No such file or directory: '../Data/crawl2_files/meta_findit_all_merged.csv'

In [138]:
#df_meta.info()
df_meta[df_meta["ID"] == 2738459452]

,abstract_ts,access_ss,Affiliations,Timestamp,Author,citation_count_i,ID,dtu_library_collection_facet,collection_facet,Publication Year,...,keywords_facet,keywords_normalized,isolanguage_facet,member_id_ss,ORCID,primary_member_id_s,Publisher,Source,source_all_ss,Title


### Pipeline 4: `LOCAL_endpoint_match_n_export.py`
file(s):


# APPENDING META FILES INTO ONE FILE

In [205]:
# Source - https://stackoverflow.com/a/55914368
# Posted by Kenry Sanchez, modified by community. See post 'Timeline' for change history
# Retrieved 2026-05-06, License - CC BY-SA 4.0

import pandas as pd
import os
meta_folder = "../Data/crawl2_files/meta_findit/raw_downloads/"
files_to_append = [os.path.join(meta_folder, file) for file in os.listdir(meta_folder) if file.endswith(".csv")]
print(f"Files to append: {files_to_append}")

opened = []

for file in files_to_append:
## you must puth header on 0 and index_col as none so you wont damage the 
#indexed later
  df = pd.read_csv(file, index_col= None, header = 0, sep=";")
  opened.append(df)

frame = pd.concat(opened, axis = 0, ignore_index = True)

output_file = "../Data/crawl2_files/meta_findit/meta_findit_all_merged.csv"
frame.to_csv(output_file, sep=";", index=False)

print(f"Merged {len(files_to_append)} files into: {output_file}")
print(f"Resulting shape: {frame.shape}")

Files to append: ['../Data/crawl2_files/meta_findit/raw_downloads/f4cdc8125f4c1cc6bb1adda59b353d57_1.csv', '../Data/crawl2_files/meta_findit/raw_downloads/962760aa426163c438227ab4bf416b90_1.csv', '../Data/crawl2_files/meta_findit/raw_downloads/576dcccbef4f5b17649d77dc156bc148_1.csv', '../Data/crawl2_files/meta_findit/raw_downloads/a11c6e2c386f26b24a962dfe7b09dbd7_1.csv', '../Data/crawl2_files/meta_findit/raw_downloads/c377ba079638cd56eb37c2eaa79b5c22_4.csv', '../Data/crawl2_files/meta_findit/raw_downloads/4af412c19e9f2e3027d4e7aa62c7ba31_1.csv', '../Data/crawl2_files/meta_findit/raw_downloads/e3891df872adcfbaeb6d7c7dea34c0a4_1.csv', '../Data/crawl2_files/meta_findit/raw_downloads/f4cdc8125f4c1cc6bb1adda59b353d57_2.csv', '../Data/crawl2_files/meta_findit/raw_downloads/f1c0821e42c6e47bb03a20eb2a379ea6_1.csv', '../Data/crawl2_files/meta_findit/raw_downloads/f37b5d56c47484910befb3dd5c7b1338_1.csv', '../Data/crawl2_files/meta_findit/raw_downloads/b698d6a5b09814864f2304623de386bb_1.csv', '..

In [206]:
new_frame = pd.concat([frame, pd.read_csv("../Data/crawl2_files/meta_findit/bulk_meta_findit_all_merged.csv", sep=";")], ignore_index=True)
#new_frame.info()
unique_ids = new_frame["ID"].nunique()
print(f"Number of unique IDs in the combined DataFrame: {unique_ids}")

output_file = "../Data/crawl2_files/meta_findit/meta_findit_all_merged_v2.csv"
new_frame.to_csv(output_file, sep=";", index=False)

print(f"Merged {len(files_to_append)} files into: {output_file}")
print(f"Resulting shape: {new_frame.shape}")

/var/folders/kb/z559lc5s7jzbc1c4j024j2w40000gn/T/ipykernel_54059/899796414.py:1: DtypeWarning: Columns (0: DOI, 1: ISBN) have mixed types. Specify dtype option on import or set low_memory=False.
  new_frame = pd.concat([frame, pd.read_csv("../Data/crawl2_files/meta_findit/bulk_meta_findit_all_merged.csv", sep=";")], ignore_index=True)


Number of unique IDs in the combined DataFrame: 15829
Merged 82 files into: ../Data/crawl2_files/meta_findit/meta_findit_all_merged_v2.csv
Resulting shape: (58726, 40)


# SOMEE TSHIT

In [190]:
import pandas as pd
from pathlib import Path

metrics_path = Path("../Data/crawl2_files/extracted_metrics_unified.parquet")
meta_path = Path("../Data/crawl2_files/meta_findit/meta_findit_all_merged_v2.csv")

df_metrics = pd.read_parquet(metrics_path)
df_meta = pd.read_csv(meta_path, sep=";", dtype=str)

#df_metrics.info()
#df_meta.info()

# listing the recods in df_metrics["filename"] that are not in df_meta["ID"]
filenames_metrics = set(df_metrics["filename"].apply(lambda x: x.split(".txt")[0]).astype(str))
filenames_meta = set(df_meta["ID"].astype(str))

missing_filenames = filenames_metrics - filenames_meta
print(f"Number of records in df_metrics that are missing in df_meta based on 'filename': {len(missing_filenames)}")
print(list(missing_filenames))

Number of records in df_metrics that are missing in df_meta based on 'filename': 46
['2737226471', '2733712278', '2390476037', '2409669662', '2385161882', '2524896212', '2737949242', '2734362965', '2358749247', '2736919894', '2733712317', '2372345846', '2372465945 2', '2854220536', '2372689921', '2738034840', '2739678436', '2385161941', '2739317014', '2735320284', '2355540125', '2385162026', '2385162130', '2734504687', '2735639715', '2372344975', '2731561909', '2390654733', '2392298458', '2395930191', '2735235803', '2734350199', '2385162129', '2385161988', '2524896194', '2372004308', '2737744429', '2737411061', '2737949240', '2734981068', '2372433012', '2385161979', '2372439310', '2856861409', '2372438112', '2733712119']


In [191]:
df_meta.columns.tolist()

['abstract_ts',
 'access_ss',
 'Affiliations',
 'Timestamp',
 'Author',
 'citation_count_i',
 'ID',
 'dtu_library_collection_facet',
 'collection_facet',
 'Publication Year',
 'Conference',
 'DOI',
 'Editor',
 'embargo_ssf',
 'format',
 'fulltext_availability_facet',
 'has_openaccess_fulltext_b',
 'holdings_ssf',
 'ISBN',
 'Journal Issue',
 'journal_issue_tsort',
 'journal_oa_model_ss',
 'Journal Page',
 'journal_page_start_tsort',
 'Journal Title',
 'journal_title_facet',
 'toc_key_s',
 'Journal Volume',
 'journal_vol_tsort',
 'keywords_ts',
 'keywords_facet',
 'keywords_normalized',
 'isolanguage_facet',
 'member_id_ss',
 'ORCID',
 'primary_member_id_s',
 'Publisher',
 'Source',
 'source_all_ss',
 'Title']

In [204]:
metrics_path = "extracted_metrics_unified.parquet"

df_metrics = load_parquet_to_df(f"{folder}{metrics_path}", na=True, verbose=False)
df_metrics.head()

# renaming "2372465945 2.txt" to "2372465945.txt" in the filename column of df_metrics
df_metrics["filename"] = df_metrics["filename"].str.replace("2372465945 2.txt", "2372465945.txt", regex=False)
df_metrics[df_metrics["filename"].str.contains("2372465945")].head()

# export the updated df_metrics to a new parquet file
output_file = "../Data/crawl2_files/extracted_metrics_unified.parquet"
df_metrics.to_parquet(output_file, index=False)